In [3]:
import pandas as pd
import re
from geotext import GeoText

# File paths
rating_file = "../../data/users-score-2023.csv" 
user_info_file = "../../data/users-details-2023.csv"  
anime_info_file = "../../data/preprocessed_anime.csv" 

# List of United-Staet for handle element that is state in US in Locatoin column.
US_STATES = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA", "Colorado": "CO",
    "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
    "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY"
}

# Reverse mapping for abbreviation lookup
US_STATES_REV = {abbr: state for state, abbr in US_STATES.items()}

# Function to detect if location is a US state
def is_us_state(location):
    if not isinstance(location, str) or pd.isna(location):
        return False

    # Check for full state names
    for state in US_STATES.keys():
        if re.search(rf"\b{state}\b", location, re.IGNORECASE):
            return True

    # Check for state abbreviations
    for abbr in US_STATES_REV.keys():
        if re.search(rf"\b{abbr}\b", location, re.IGNORECASE):
            return True

    return False

# Function to process Location: Convert US states to "United States", others to country
def process_location(location):
    if is_us_state(location):
        return "United States"  # Convert all US states to "United States"

    # Use geotext to extract country
    places = GeoText(location)
    country = places.countries
    if country:
        return country[0]  # Return country name

    return "Unknown"  # If nothing is detected

# handle genre issue
def split_genres(df, column="Genres"):
    df[column] = df[column].fillna("")  # Handle NaN values
    df[column] = df[column].str.split(", ")  # Split genres into lists
    df = df.explode(column, ignore_index=True)  # Create separate rows for each genre
    return df

# Step 1: Read and process the ratings CSV (keep only the first rating per user)
rating_df = pd.read_csv(rating_file)
rating_df = rating_df.sort_values(by=["user_id", "anime_id"])  # Sort to keep the first occurrence
rating_df = rating_df.drop_duplicates(subset=["user_id"], keep="first")  # Keep first rating per user

# Step 2: Read the user info CSV
user_info_df = pd.read_csv(user_info_file)

if "Mal ID" in user_info_df.columns:
    user_info_df.rename(columns={"Mal ID": "user_id"}, inplace=True)

# Step 3: Read the anime info CSV
anime_info_df = pd.read_csv(anime_info_file)

# Step 4: Merge rating_df with user_info_df on UserID
merged_df = pd.merge(rating_df, user_info_df, on="user_id", how="left")

# Step 5: Merge the result with anime_info_df on AnimeID
merged_df = pd.merge(merged_df, anime_info_df, on="anime_id", how="left")

# remove the NAN
merged_df = merged_df.dropna()

# convert the Location column
merged_df["Location"] = merged_df["Location"].apply(process_location)

# sampling the data for smaller size
merged_df = merged_df.sample(n = 8000, random_state=6)

# remove the Location that is Unknown
merged_df = merged_df[merged_df["Location"] != "Unknown"]

# split the genre
merged_df = split_genres(merged_df, "Genres")

# Step 6: Save the final dataset
merged_df.to_csv("../../data/merged_data.csv", index=False)


In [38]:
## read the merged file
import pandas as pd
merged_file = pd.read_csv("merged_data.csv")

# check number of records
print(merged_file.shape[0])

254251


In [45]:
merged_file.head(10)

,user_id,Username_x,anime_id,Anime Title,rating,Username_y,Gender,Birthday,Location,Joined,...,Ranked,Popularity,Members,Favorites,Watching_y,Completed_y,On-Hold,Dropped_y,start_date,end_date
0,1,Xinil,1,Cowboy Bebop,10,Xinil,Male,1985-03-04T00:00:00+00:00,United States,2004-11-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
1,1,Xinil,1,Cowboy Bebop,10,Xinil,Male,1985-03-04T00:00:00+00:00,United States,2004-11-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
2,1,Xinil,1,Cowboy Bebop,10,Xinil,Male,1985-03-04T00:00:00+00:00,United States,2004-11-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
3,1,Xinil,1,Cowboy Bebop,10,Xinil,Male,1985-03-04T00:00:00+00:00,United States,2004-11-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
4,1,Xinil,1,Cowboy Bebop,10,Xinil,Male,1985-03-04T00:00:00+00:00,United States,2004-11-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
5,1,Xinil,1,Cowboy Bebop,10,Xinil,Male,1985-03-04T00:00:00+00:00,United States,2004-11-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
6,20,vondur,1,Cowboy Bebop,9,vondur,Male,1988-01-25T00:00:00+00:00,Norway,2005-01-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
7,20,vondur,1,Cowboy Bebop,9,vondur,Male,1988-01-25T00:00:00+00:00,Norway,2005-01-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
8,20,vondur,1,Cowboy Bebop,9,vondur,Male,1988-01-25T00:00:00+00:00,Norway,2005-01-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24
9,20,vondur,1,Cowboy Bebop,9,vondur,Male,1988-01-25T00:00:00+00:00,Norway,2005-01-05T00:00:00+00:00,...,28.0,39.0,1251960.0,61971.0,105808.0,718161.0,71513.0,26678.0,1998-04-03,1999-04-24


In [56]:
df = merged_df.copy()
df_grouped = df.groupby(["Location", "Genres", "Type", "Source"])["user_id"].nunique().reset_index()
df_grouped.rename(columns={"user_id": "Total Viewers"}, inplace=True)

In [ ]:
# Calculate total viewers per Location
total_viewers_by_location = df_grouped.groupby("Location")["Total Viewers"].sum().reset_index()
total_viewers_by_location.rename(columns={"Total Viewers": "Total Viewers by Location"}, inplace=True)

,Location,Total Viewers by Location
0,Afghanistan,2
1,Albania,1
2,Algeria,9
3,Angola,2
4,Antarctica,8
